10강

https://youtu.be/iD0FbossURg?si=M5ztj9QQ8ySdjDRb

# 딥러닝의 고수가 되기 위해 알아야 하는 모듈들

# nn의 유용한 모듈들 .parameters() & .modules() & .children()

In [ ]:
import torch
from torch import nn

In [ ]:
class MLP(nn.Module): # 3층짜리 이진분류의 간단한 신경망
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Sequential(nn.Linear(2,3),
                                 nn.ReLU())
        self.fc2 = nn.Sequential(nn.Linear(3,4),
                                 nn.ReLU())
        self.fc_out = nn.Sequential(nn.Linear(4,1),
                                    nn.Sigmoid())
    def forward(self,x):
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc_out(x)
        return x

model = MLP()
print(model(torch.randn(2,2)).shape)
print(model)

torch.Size([2, 1])
MLP(
  (fc1): Sequential(
    (0): Linear(in_features=2, out_features=3, bias=True)
    (1): ReLU()
  )
  (fc2): Sequential(
    (0): Linear(in_features=3, out_features=4, bias=True)
    (1): ReLU()
  )
  (fc_out): Sequential(
    (0): Linear(in_features=4, out_features=1, bias=True)
    (1): Sigmoid()
  )
)


## 파라미터 개수를 구할 때 + 특정 파라미터만 수정하고 싶을때? -> model.parameters()

In [ ]:
model.parameters()

<generator object Module.parameters at 0x7ab9d1afbd80>

In [ ]:
list(model.parameters())

[Parameter containing:
 tensor([[-0.3317,  0.3209],
         [-0.2673,  0.5655],
         [ 0.5209, -0.1152]], requires_grad=True),
 Parameter containing:
 tensor([0.6919, 0.1810, 0.2472], requires_grad=True),
 Parameter containing:
 tensor([[-0.0119,  0.2912,  0.4720],
         [-0.1311, -0.1241,  0.0302],
         [ 0.0078,  0.1545,  0.5317],
         [-0.2908, -0.3557, -0.5494]], requires_grad=True),
 Parameter containing:
 tensor([ 0.0186, -0.0074,  0.4114,  0.1504], requires_grad=True),
 Parameter containing:
 tensor([[ 0.2154, -0.0854,  0.4232, -0.0914]], requires_grad=True),
 Parameter containing:
 tensor([0.3111], requires_grad=True)]

- 파라미터를 볼때 Affine의 w tensor 하나, b tensor 하나, 이렇게 순차적으로 번걸아가면서 나옴.

In [ ]:
list(model.parameters())[3] # 개별 레이어로 보기
# [layer0 weight 값, layer0 bias 값, layer1 weight 값, layer1 bias 값, ...]

Parameter containing:
tensor([ 0.0186, -0.0074,  0.4114,  0.1504], requires_grad=True)

- 아래 처럼 이름과 함께 출력하게 만들면 보기 편함

In [ ]:
for name, param in model.named_parameters():
    print(f"이름: {name}")
    print(f"형태: {param.shape}")
    print("-" * 20)

이름: fc1.0.weight
형태: torch.Size([3, 2])
--------------------
이름: fc1.0.bias
형태: torch.Size([3])
--------------------
이름: fc2.0.weight
형태: torch.Size([4, 3])
--------------------
이름: fc2.0.bias
형태: torch.Size([4])
--------------------
이름: fc_out.0.weight
형태: torch.Size([1, 4])
--------------------
이름: fc_out.0.bias
형태: torch.Size([1])
--------------------


- 파라미터 보는게 transfer learning에 쓰임.

In [ ]:
# for transfer learning
model = MLP()
print([p for p in model.parameters() if p.requires_grad]) # requires_grad True인 애들 가져와서 보기

print('='*50)

for p in model.parameters(): # 전체 freeze # requires_grad를 False로
    p.requires_grad = False

# 마지막 층만 오버라이딩!
model.fc_out = nn.Linear(4,10) # 이진 분류에서 열 개 분류로!

# 다시 출력해서 확인
params = [p for p in model.parameters() if p.requires_grad]
print(params) # 바뀜! 마지막 새로 추가된 4 입력 10 출력 레이어의 파라미터만 있음

from torch import optim
optimizer = optim.Adam(params, lr=0.1)

[Parameter containing:
tensor([[-0.1117, -0.0195],
        [-0.5519,  0.0896],
        [ 0.0247, -0.3947]], requires_grad=True), Parameter containing:
tensor([-0.1306,  0.2025, -0.3921], requires_grad=True), Parameter containing:
tensor([[-0.3472, -0.2287, -0.2637],
        [ 0.1775, -0.2876, -0.2310],
        [-0.1161,  0.5661,  0.2822],
        [-0.0165,  0.5530, -0.0106]], requires_grad=True), Parameter containing:
tensor([0.0275, 0.4301, 0.4302, 0.0165], requires_grad=True), Parameter containing:
tensor([[ 0.3857, -0.1728,  0.3660,  0.3829]], requires_grad=True), Parameter containing:
tensor([0.3865], requires_grad=True)]
[Parameter containing:
tensor([[ 0.0801,  0.4990, -0.2680, -0.0875],
        [ 0.0741,  0.1074,  0.4756,  0.0618],
        [ 0.4222, -0.1665,  0.4438, -0.4904],
        [-0.2898, -0.0054, -0.0295, -0.0007],
        [ 0.3031, -0.1167,  0.4511,  0.0908],
        [ 0.0836, -0.3433,  0.4560, -0.2185],
        [ 0.0163, -0.4876, -0.3127, -0.3267],
        [-0.2770, -0.

In [ ]:
# for tranfer learning (마지막 층은 교체, 그 전 층은 학습시키자)
model = MLP()
print([p for p in model.parameters() if p.requires_grad])

print('='*50)

for p in list(model.parameters())[:-4]: # 마지막 전 층 전까지만 freeze +
    # (그러니까 입력 쪽은 freeze, 출력 쪽에서 부터 4개는 업데이트 됨)
    # 이게 왜 -4? <- w,b 각각 tensor로 나와서, -4은 w,b,w,b 텐서 이렇게 2층을 업데이트하는겨.
    p.requires_grad = False
model.fc_out = nn.Linear(4,10) # 이진 분류에서 열 개 분류로!

params = [p for p in model.parameters() if p.requires_grad]
print(params)

from torch import optim
optimizer = optim.Adam(params, lr=0.1)

[Parameter containing:
tensor([[-0.5406, -0.1677],
        [-0.2832, -0.1728],
        [-0.0524,  0.5866]], requires_grad=True), Parameter containing:
tensor([-0.3046, -0.2819,  0.2147], requires_grad=True), Parameter containing:
tensor([[-0.1798, -0.4684,  0.2153],
        [ 0.1815, -0.1567, -0.4984],
        [-0.5539,  0.1573,  0.1209],
        [-0.0143,  0.5570, -0.2306]], requires_grad=True), Parameter containing:
tensor([ 0.1087, -0.3846,  0.4103,  0.5205], requires_grad=True), Parameter containing:
tensor([[-0.4935, -0.4411,  0.3434, -0.1464]], requires_grad=True), Parameter containing:
tensor([0.1243], requires_grad=True)]
[Parameter containing:
tensor([[-0.1798, -0.4684,  0.2153],
        [ 0.1815, -0.1567, -0.4984],
        [-0.5539,  0.1573,  0.1209],
        [-0.0143,  0.5570, -0.2306]], requires_grad=True), Parameter containing:
tensor([ 0.1087, -0.3846,  0.4103,  0.5205], requires_grad=True), Parameter containing:
tensor([[-0.4528, -0.2842,  0.3357, -0.0631],
        [-0.1

In [ ]:
list(model.named_parameters())[1]
# [('layer0.weight', weight 값), ('layer0.bias', bias 값), ('layer1.weight', weight 값), ('layer1.bias', bias 값), ...] <= 튜플 품은 리스트

('fc1.0.bias',
 Parameter containing:
 tensor([-0.3046, -0.2819,  0.2147]))

In [ ]:
for name, p in model.named_parameters():
    print(name)
    print(p)

fc1.0.weight
Parameter containing:
tensor([[-0.5406, -0.1677],
        [-0.2832, -0.1728],
        [-0.0524,  0.5866]])
fc1.0.bias
Parameter containing:
tensor([-0.3046, -0.2819,  0.2147])
fc2.0.weight
Parameter containing:
tensor([[-0.1798, -0.4684,  0.2153],
        [ 0.1815, -0.1567, -0.4984],
        [-0.5539,  0.1573,  0.1209],
        [-0.0143,  0.5570, -0.2306]], requires_grad=True)
fc2.0.bias
Parameter containing:
tensor([ 0.1087, -0.3846,  0.4103,  0.5205], requires_grad=True)
fc_out.weight
Parameter containing:
tensor([[-0.4528, -0.2842,  0.3357, -0.0631],
        [-0.1075, -0.3630, -0.2922,  0.2704],
        [ 0.4553,  0.1910, -0.4964, -0.2757],
        [-0.0609, -0.1090, -0.3999, -0.4618],
        [-0.3620,  0.3199, -0.4520, -0.4055],
        [ 0.0481,  0.2942, -0.3028, -0.3724],
        [ 0.1250,  0.2188, -0.3567,  0.3215],
        [ 0.2289,  0.4687,  0.0826, -0.2245],
        [-0.2899,  0.3852, -0.1505, -0.3005],
        [ 0.1556,  0.3801,  0.4170,  0.0343]], requires_gra

## model.modules()

- 제너레이터임

In [ ]:
model.modules()

<generator object Module.modules at 0x7ab9d496ea80>

- modules는 모델이 가진 모든 모듈을 불러옴

In [ ]:
list(model.modules())

[MLP(
   (fc1): Sequential(
     (0): Linear(in_features=2, out_features=3, bias=True)
     (1): ReLU()
   )
   (fc2): Sequential(
     (0): Linear(in_features=3, out_features=4, bias=True)
     (1): ReLU()
   )
   (fc_out): Linear(in_features=4, out_features=10, bias=True)
 ),
 Sequential(
   (0): Linear(in_features=2, out_features=3, bias=True)
   (1): ReLU()
 ),
 Linear(in_features=2, out_features=3, bias=True),
 ReLU(),
 Sequential(
   (0): Linear(in_features=3, out_features=4, bias=True)
   (1): ReLU()
 ),
 Linear(in_features=3, out_features=4, bias=True),
 ReLU(),
 Linear(in_features=4, out_features=10, bias=True)]

- 위 출력을 잘 보면 () 단위로 하나씩 까서 그 속의 모듈들의 구조를 다 보여줌. 근데 구조가 약간 특이함.
  - 일단 맨 위에서 **전체**를 다 보여줌. (Sequential 이름별로)
  - 그 다음 단계에서 안에 있던 Sequential **단독으로 하나**씩 보여줌.
    - 각 Sequential 바로 다음에, 그 Sequential **속에 있던 모듈**들도 꺼내서 보여줌.

- 아래 처럼 모듈별로 분류해서 볼 수 있음. (나중에 **모듈별로 뭔가를 수정**하고 싶을 때 유용함)

In [ ]:
print([m for m in model.modules() if isinstance(m,nn.Linear)])
print([m.weight for m in model.modules() if isinstance(m,nn.Linear)])
print([m.weight.grad for m in model.modules() if isinstance(m,nn.Linear)])

[Linear(in_features=2, out_features=3, bias=True), Linear(in_features=3, out_features=4, bias=True), Linear(in_features=4, out_features=10, bias=True)]
[Parameter containing:
tensor([[-0.5406, -0.1677],
        [-0.2832, -0.1728],
        [-0.0524,  0.5866]]), Parameter containing:
tensor([[-0.1798, -0.4684,  0.2153],
        [ 0.1815, -0.1567, -0.4984],
        [-0.5539,  0.1573,  0.1209],
        [-0.0143,  0.5570, -0.2306]], requires_grad=True), Parameter containing:
tensor([[-0.4528, -0.2842,  0.3357, -0.0631],
        [-0.1075, -0.3630, -0.2922,  0.2704],
        [ 0.4553,  0.1910, -0.4964, -0.2757],
        [-0.0609, -0.1090, -0.3999, -0.4618],
        [-0.3620,  0.3199, -0.4520, -0.4055],
        [ 0.0481,  0.2942, -0.3028, -0.3724],
        [ 0.1250,  0.2188, -0.3567,  0.3215],
        [ 0.2289,  0.4687,  0.0826, -0.2245],
        [-0.2899,  0.3852, -0.1505, -0.3005],
        [ 0.1556,  0.3801,  0.4170,  0.0343]], requires_grad=True)]
[None, None, None]


- 예를 들어 특정 레이러들을 원하는 값으로 초기화하고 싶다고하자. (**초깃값 초기화**)

In [ ]:
# weight initialization에 활용
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight) # kaiming(He 변형)으로 초기화 ++
        # nn.init.constant_(m.weight, 1) # 그냥 뒤 1 값으로 다 초기화

print([m.weight for m in model.modules() if isinstance(m, nn.Linear)])

[Parameter containing:
tensor([[ 0.8242, -0.1909],
        [ 0.9288,  0.1442],
        [ 0.0165,  0.1455]]), Parameter containing:
tensor([[ 0.0448,  0.3565, -0.2622],
        [ 0.2759, -1.7574,  0.5379],
        [ 0.8067, -0.3334,  1.1448],
        [ 0.0193, -0.0663, -0.2620]], requires_grad=True), Parameter containing:
tensor([[ 0.0695, -0.8170,  0.0147, -0.1714],
        [-1.0931,  1.4637, -0.7584, -0.4675],
        [-0.3086, -0.6653,  0.1078, -0.2381],
        [-0.8223,  1.1093, -0.1473, -0.3056],
        [-0.2303,  0.0458,  0.0665, -0.2041],
        [-0.9701,  0.7294,  0.2191,  0.6362],
        [ 0.1068, -0.6094, -0.0325, -0.4873],
        [-0.3565, -0.9120,  0.5350,  0.5030],
        [-0.5854, -0.1880, -0.6514, -0.6764],
        [ 1.0431,  0.0940,  0.0361, -0.1789]], requires_grad=True)]


- 파이토치에서 함수 이름 맨 끝에 언더바(_)가 붙어있으면, 새로운 값을 반환하는 대신 원본 텐서의 값을 **직접 수정한다**(**In-place** operation)는 뜻

  kaiming_normal_()

- 참고로 nn init 검색해서 보면 여러 초기화 기법들을 볼 수 있음. (Xavier, He 등등)

https://docs.pytorch.org/docs/stable/nn.init.html

## model.children()


- 마찬가지로 제너레이터

In [ ]:
model.children()

<generator object Module.children at 0x7ab9d496eb50>

- 이 친구는 Attributes(속성)을 보여주는 느낌? modules랑 비슷함.

In [ ]:
list(model.children())

[Sequential(
   (0): Linear(in_features=2, out_features=3, bias=True)
   (1): ReLU()
 ),
 Sequential(
   (0): Linear(in_features=3, out_features=4, bias=True)
   (1): ReLU()
 ),
 Linear(in_features=4, out_features=10, bias=True)]

- 놀랍게도 아래 처럼 **그 부분만 통과**하는 응용이 가능함.

In [ ]:
x = torch.randn(2,2) # 랜덤한 데이터 만들어서
list(model.children())[0](x) # 이 한층만 통과시키기 ++

tensor([[1.2435, 1.7790, 0.3771],
        [0.0000, 0.0000, 0.0975]])

- 물론 modules과 다르게 Sequential 단위로만 접근이 되기에 초기화에는 애매하고, 아래 처럼 커다란 **신경망의 일부분**을 사용해서(**서브 네트워크**) 무언가를 할 때 유용.

In [ ]:
sub_network = nn.Sequential(*list(model.children())[:2])
print(sub_network)
sub_network(x)

Sequential(
  (0): Sequential(
    (0): Linear(in_features=2, out_features=3, bias=True)
    (1): ReLU()
  )
  (1): Sequential(
    (0): Linear(in_features=3, out_features=4, bias=True)
    (1): ReLU()
  )
)


tensor([[0.6996, 0.0000, 1.2519, 0.3279],
        [0.0831, 0.0000, 0.5219, 0.4950]], grad_fn=<ReluBackward0>)

---
10-2강

https://youtu.be/TAYdlgJt9KM?si=5kDMEJfGxxMpQnKs

# Sequential vs ModuleList

그동안 Sequential만을 사용했는데, **트랜스포머** 같은거 공부하다보면 **ModuleList** 같은 친구도 나옴.

In [ ]:
fc=nn.Linear(3,3) # 33 레이어를 하나 만들고
layer_list = [fc for _ in range(5)] # 이를 5번 반복해서 리스트에 집어넣기

# 만약 이를 Sequential에 넣으려면 언패킹을 해야함
layers1 = nn.Sequential(*layer_list)
# Modulelist는 리스트 형식으로 집어넣어도 됨
layers2 = nn.ModuleList(layer_list)

# 보면 출력 되는게 약간 다르다는 것을 볼 수 있음
print(layers1)
print(layers2)

x=torch.randn(1,3) # 인풋 데이터 생성

# 신경망에 통과 시킬 때 Sequential은 그냥 이렇게 하면 되지만
print(layers1(x))

# Modulelist는 이렇게 하면 에러 뜸!
# print(layers2(x)) # error!
# 반복문을 통해서 하나씩 꺼내서 다 통과해줘야함! +++
for layer in layers2:
    x = layer(x)
print(x)

Sequential(
  (0): Linear(in_features=3, out_features=3, bias=True)
  (1): Linear(in_features=3, out_features=3, bias=True)
  (2): Linear(in_features=3, out_features=3, bias=True)
  (3): Linear(in_features=3, out_features=3, bias=True)
  (4): Linear(in_features=3, out_features=3, bias=True)
)
ModuleList(
  (0-4): 5 x Linear(in_features=3, out_features=3, bias=True)
)
tensor([[-1.0958,  0.3364, -0.9288]], grad_fn=<AddmmBackward0>)
tensor([[-1.0958,  0.3364, -0.9288]], grad_fn=<AddmmBackward0>)


- 그러니까 Sequential은 모든 레이어를 하나의 신경망으로 합쳐주지만!
- ModuleList 말그대로 레이어들의 리스트지, 내부에서 합쳐져 있지 않음!

### 근데 하는 역할을 비슷한데 ModuleList가 더 불편한데 왜 씀?

- 이유는 **세세하게** 입력과 출력을 조절할 때 Sequential을 쓰면 불편함.

- 예를 들어 입력이 2개고, 안에서 병렬로 신경망이 돌아가고, 출력이 2개인 신경망을 Sequential은 간단하게 못 만듬. `forward`를 오버라이딩 해서 수정해야함.

In [ ]:
# 그럼 nn.Sequential 쓰고 말지 왜 굳이 nn.ModuleList?
class small_block(nn.Module):
    def __init__(self):
        super().__init__()
        self.block_x = nn.Linear(1,1)
        self.block_y = nn.Linear(1,1)

    def forward(self, x, y):
        x = self.block_x(x)
        y = self.block_y(y)
        return x, y

model = nn.Sequential(small_block(),
                      small_block())
# model(torch.randn(1),torch.randn(1)) # error 발생! 입력이 잘못됨!
# nn.Sequential 이 가지고 있는 forward 함수를 call 하기 때문에 입력을 두 개 넣으면 안 된다!! ++

# 근데 ModuleList를 쓰면? +++
model = nn.ModuleList([small_block(),
                       small_block()])
x = torch.randn(1)
y = torch.randn(1)
for block in model:
    x, y = block(x,y) # 매우 간단스 +
print(x, y)
# 유동적으로 쓰기 좋음

tensor([-0.8085], grad_fn=<ViewBackward0>) tensor([0.0176], grad_fn=<ViewBackward0>)


- ModuleList를 클래스에서 사용하면?

In [ ]:
# 걍 리스트 쓰지 왜 nn.ModuleList 를 쓸까?
class testNet(nn.Module):
    def __init__(self):
        super().__init__()

        # self.Module_List = [nn.Linear(3,3), nn.Linear(3,3)] # 이렇게 넣으면 당연히 안됨, 아래에서 에러남
        self.Module_List = nn.ModuleList([nn.Linear(3,3), nn.Linear(3,3)])

    def forward(self,x):
        for layer in self.Module_List:
            x = layer(x)
        return x

model=testNet()
print(model(torch.randn(1,3))) # 통과까진 문제 없음

print(model) # 그냥 리스트로 하니 아무것도 등록이 안 돼 있다!

optimizer = optim.Adam(model.parameters(), lr = 0.1) # 등록이 안 돼 있으면 parameter를 못 찾는다!

tensor([[-0.2393, -0.0249,  0.6082]], grad_fn=<AddmmBackward0>)
testNet(
  (Module_List): ModuleList(
    (0-1): 2 x Linear(in_features=3, out_features=3, bias=True)
  )
)
